# Qwen3.5-9B · 512² · 양자화 없는 BF16 LoRA

**요청 반영: 픽셀 예산 384² → 512², NF4 4bit → 원본 BF16 가중치.**

기존 베이스라인과 같은 폴더에 넣고, 충분한 VRAM의 CUDA 커널에서 Run All을 실행하세요.
데이터 확인 → GPU 메모리 확인 → 필요한 패키지/원본 모델 준비 → LoRA 학습 → 검증 loss → 전체 test 추론 → 제출 CSV 순서입니다.

**RTX 5060 Ti 16GB에서는 실행할 수 없습니다.** 9B 파라미터의 16bit 가중치만 약 18GB 이상이고 활성값과 학습 메모리가 더 필요합니다.
16GB GPU에서는 다운로드·학습 전에 이유를 표시하고 중단합니다. 더 큰 GPU에서도 실제 학습 VRAM은 미측정이며 실행을 보장하지 않습니다.
CPU offload 또는 4bit로 자동 전환하지 않습니다. GPU 대여·환경 변경은 이 파일이 수행하지 않습니다.

- 원본 BF16 가중치를 직접 로드합니다. BF16 미지원 GPU에서는 비양자화 FP16을 사용하며 메모리 절약 효과는 없습니다.
- 기존 `downloads/models/Qwen3.5-9B/`가 완성돼 있으면 그대로 재사용합니다. 없으면 원본 모델을 자동 다운로드합니다.
- 유지: 23셀, seed42, 200→180/20, batch1/accum4, LoRA r8/alpha16/dropout.05, lr1e-4, epoch1, 전체 토큰 loss, 기존 프롬프트·파서·greedy2토큰, non-thinking.
- 이미지 종횡비는 processor가 처리합니다. 512×512 강제 변형이 아니라 min/max 픽셀 예산을 512²로 변경합니다.
- 양자화 해제 후에도 기본 가중치는 동결하고 LoRA만 새로 학습합니다. 전체 파라미터 미세조정이 아닙니다.
- 결과: `output/TASK-003/EXP-010/<실행시각>/submission.csv`, `qwen3_5_9b_lora/`, `run_config.json`, `split_manifest.csv`.
- 기준 0.83은 이전 NF4/384² 버전의 사용자 보고값입니다. 이 버전의 점수는 아직 없습니다.

해상도와 정밀도를 함께 변경한 실험이므로 두 변경의 개별 기여는 분리할 수 없습니다.
기존 0.83 어댑터를 이어서 학습하는 것이 아니라 같은 원본 모델에서 새 LoRA를 학습합니다.
커널에 이미 불러온 패키지가 교체되면 안내에 따라 Restart Kernel → Run All을 실행하세요.

TASK-003 / EXP-010 / v1.3. 코드 반영·로컬 점검 완료, 실제 GPU 실행 전.

# 자동 환경 준비

기존 CUDA PyTorch를 유지하고 필요한 패키지만 설치합니다. GPU 드라이버는 자동 변경하지 않습니다.
CUDA 및 단일 GPU의 가중치 적재 최소 용량을 먼저 확인합니다. 이 검사는 전체 학습 메모리를 보장하지 않습니다.
부족한 메모리·데이터·패키지 오류가 있으면 중단합니다.

In [1]:
import os, sys, json, subprocess, importlib, importlib.metadata as metadata
from pathlib import Path

print("Run All: 환경 준비 → 모델 다운로드 → 학습 → 추론 → 제출 파일 생성")
# 이후 다운로드는 새 Python 프로세스에서 실행하여, 이전 커널의 HF 오프라인 상수 캐시와 분리합니다.

Run All: 환경 준비 → 모델 다운로드 → 학습 → 추론 → 제출 파일 생성


In [2]:
ASSET_DIR = "downloads"
LIB_DIR = os.path.join(ASSET_DIR, "libs")
MODEL_REPO_ID = "Qwen/Qwen3.5-9B"
MODEL_DIR = os.path.join(ASSET_DIR, "models", "Qwen3.5-9B")
DATA_DIR = "data"
from datetime import datetime
OUTPUT_DIR = os.path.join("output", "TASK-003", "EXP-010", datetime.now().strftime("%Y%m%d_%H%M%S_%f"))

for filename in ["train.csv", "test.csv"]:
    if not os.path.isfile(os.path.join(DATA_DIR, filename)):
        raise FileNotFoundError(f"{DATA_DIR}/{filename}가 없습니다. 기존 베이스라인 폴더에서 실행하세요.")
for folder in ["train", "test"]:
    if not os.path.isdir(os.path.join(DATA_DIR, folder)):
        raise FileNotFoundError(f"{DATA_DIR}/{folder}/ 이미지 폴더가 없습니다.")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print("모델:", MODEL_REPO_ID)
print("결과 폴더:", os.path.abspath(OUTPUT_DIR))

모델: Qwen/Qwen3.5-9B
결과 폴더: c:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-003\EXP-010\20260921_141410_411059


In [3]:
# 기존 CUDA PyTorch를 유지. 없을 때만 원본 CUDA 빌드를 설치합니다.
try:
    torch_version = metadata.version("torch")
    print("기존 PyTorch 유지:", torch_version)
except metadata.PackageNotFoundError:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--index-url", "https://download.pytorch.org/whl/cu128",
        "torch==2.11.0+cu128", "torchvision==0.26.0+cu128"
    ], check=True)
    importlib.invalidate_caches()
    print("CUDA PyTorch 설치 완료")

기존 PyTorch 유지: 2.11.0+cu128


In [ ]:
# 커널에 torch를 먼저 import하지 않도록 별도 프로세스로 확인합니다.
# 9B × 2 bytes는 가중치만의 보수적인 하한이며, 학습에는 추가 메모리가 필요합니다.
GPU_CHECK_CODE = """
import torch
assert torch.cuda.is_available(), 'CUDA GPU가 필요합니다.'
props = torch.cuda.get_device_properties(0)
minimum_weight_bytes = 9_000_000_000 * 2
print('PyTorch:', torch.__version__)
print('GPU:', props.name)
print('VRAM:', round(props.total_memory / 1024**3, 1), 'GiB')
if props.total_memory <= minimum_weight_bytes:
    raise RuntimeError(
        'Qwen3.5-9B 비양자화 BF16/FP16 가중치만 약 18GB 이상입니다. '
        '현재 GPU에는 가중치가 들어가지 않습니다. 5060 Ti 16GB에서는 이 버전을 실행할 수 없습니다. '
        '충분한 VRAM의 GPU가 필요하며, 기존 4bit 버전은 별도 파일로 보존되어 있습니다.'
    )
print('가중치 최소 용량 점검 통과. 전체 학습 VRAM의 실행 보장은 아닙니다.')
"""
try:
    subprocess.run([sys.executable, "-c", GPU_CHECK_CODE], check=True)
except subprocess.CalledProcessError as exc:
    with open(os.path.join(OUTPUT_DIR, "setup_failure.json"), "w", encoding="utf-8") as f:
        json.dump({"task_id": "TASK-003", "experiment_id": "EXP-010",
                   "stage": "gpu_preflight", "error": str(exc),
                   "detail": "위 CUDA/VRAM 오류를 확인하세요. 양자화/CPU offload 자동 전환 없음."},
                  f, ensure_ascii=False, indent=2)
    raise

CalledProcessError: Command '['c:\\Users\\SSAFY\\Desktop\\AI2_Challenge\\baseline\\Scripts\\python.exe', '-c', "\nimport torch\nassert torch.cuda.is_available(), 'CUDA GPU가 필요합니다.'\nprops = torch.cuda.get_device_properties(0)\nminimum_weight_bytes = 9_000_000_000 * 2\nprint('PyTorch:', torch.__version__)\nprint('GPU:', props.name)\nprint('VRAM:', round(props.total_memory / 1024**3, 1), 'GiB')\nif props.total_memory <= minimum_weight_bytes:\n    raise RuntimeError(\n        'Qwen3.5-9B 비양자화 BF16/FP16 가중치만 약 18GB 이상입니다. '\n        '현재 GPU에는 가중치가 들어가지 않습니다. 5060 Ti 16GB에서는 이 버전을 실행할 수 없습니다. '\n        '충분한 VRAM의 GPU가 필요하며, 기존 4bit 버전은 별도 파일로 보존되어 있습니다.'\n    )\nprint('가중치 최소 용량 점검 통과. 전체 학습 VRAM의 실행 보장은 아닙니다.')\n"]' returned non-zero exit status 1.

: 

In [ ]:
# 1) 필요한 의존성만 설치. 이미 설치된 CUDA torch 버전은 pip constraint로 고정.
import tempfile
try:
    from packaging.requirements import Requirement
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "packaging"], check=True)
    from packaging.requirements import Requirement

requirements = ["transformers>=5.8.0,<6.0.0", "peft>=0.18.0", "accelerate>=0.34.2",
                "huggingface_hub", "pandas", "Pillow", "tqdm", "torchvision"]
def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

def missing_requirements(requirements):
    missing = []
    for value in requirements:
        req = Requirement(value)
        version = installed_version(req.name)
        if version is None or (req.specifier and not req.specifier.contains(version, prereleases=True)):
            missing.append(value)
    return missing

needed = missing_requirements(requirements)
if needed:
    watched = {"torch": "torch", "torchvision": "torchvision", "transformers": "transformers",
               "peft": "peft", "accelerate": "accelerate", 
               "huggingface_hub": "huggingface_hub", "tokenizers": "tokenizers", "numpy": "numpy",
               "PIL": "Pillow", "pandas": "pandas", "safetensors": "safetensors"}
    loaded_before = {dist: installed_version(dist) for module, dist in watched.items() if module in sys.modules}
    with tempfile.TemporaryDirectory() as temp_dir:
        constraint = Path(temp_dir) / "keep_torch.txt"
        constraint.write_text("torch==" + metadata.version("torch") + "\n", encoding="utf-8")
        print("필요한 패키지 설치:", needed)
        subprocess.run([sys.executable, "-m", "pip", "install", "--constraint", str(constraint), *needed], check=True)
    importlib.invalidate_caches()
    changed_loaded = [dist for dist, old in loaded_before.items() if installed_version(dist) != old]
    if changed_loaded:
        raise RuntimeError(f"이미 불러온 패키지가 변경됐습니다: {changed_loaded}. Restart Kernel → Run All을 한 번 실행하세요.")
if missing_requirements(requirements):
    raise RuntimeError("필요한 패키지 버전이 설치되지 않았습니다. 위 설치 로그를 확인하세요.")

# 2) 완료된 모델은 재사용. 설정/토크나이저/인덱스와 모든 shard의 헤더·길이를 확인합니다.
# 파일 전체의 암호학적 무결성을 검증하는 것은 아닙니다.
def local_model_complete(model_dir):
    folder = Path(model_dir)
    required = ["config.json", "tokenizer_config.json", "tokenizer.json", "preprocessor_config.json",
                "video_preprocessor_config.json", "chat_template.jinja", "model.safetensors.index.json"]
    try:
        if any(not (folder / name).is_file() or (folder / name).stat().st_size == 0 for name in required):
            return False
        for name in required:
            if name.endswith(".json"):
                json.loads((folder / name).read_text(encoding="utf-8"))
        config = json.loads((folder / "config.json").read_text(encoding="utf-8"))
        if config.get("model_type") != "qwen3_5" or config.get("text_config", {}).get("hidden_size") != 4096:
            raise ValueError("MODEL_DIR의 config가 Qwen3.5-9B와 다릅니다. 경로를 확인하세요.")
        weight_map = json.loads((folder / "model.safetensors.index.json").read_text(encoding="utf-8"))["weight_map"]
        if not weight_map:
            return False
        for name in set(weight_map.values()):
            path = folder / name
            if not path.is_file():
                return False
            with path.open("rb") as stream:
                header_length = int.from_bytes(stream.read(8), "little")
                if not 0 < header_length <= min(100_000_000, path.stat().st_size-8):
                    return False
                header = json.loads(stream.read(header_length))
            offsets = [item["data_offsets"][1] for key, item in header.items() if key != "__metadata__"]
            if not offsets or max(offsets) != path.stat().st_size - 8 - header_length:
                return False
            if any(key not in header for key, shard in weight_map.items() if shard == name):
                return False
        return True
    except (OSError, KeyError, TypeError, json.JSONDecodeError, UnicodeDecodeError):
        return False

DOWNLOAD_CODE = r"""
import json, sys
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
repo_id, local_dir = sys.argv[1:3]
folder = Path(local_dir)
record_path = folder / "download_revision.json"
record = json.loads(record_path.read_text(encoding="utf-8")) if record_path.is_file() else {}
revision = record.get("revision") if record.get("repo_id") == repo_id else None
if not revision:
    revision = HfApi().model_info(repo_id).sha
record_path.write_text(json.dumps({"repo_id": repo_id, "revision": revision}, indent=2), encoding="utf-8")
snapshot_download(repo_id=repo_id, revision=revision, local_dir=local_dir,
                  allow_patterns=["*.json", "*.safetensors", "*.txt", "*.jinja", "LICENSE"], max_workers=4)
print("모델 다운로드 완료:", local_dir)
"""

def ensure_model(model_dir, repo_id, runner=None):
    runner = subprocess.run if runner is None else runner
    Path(model_dir).mkdir(parents=True, exist_ok=True)
    if local_model_complete(model_dir):
        print("다운로드 완료된 모델 재사용:", model_dir)
        return "reused"
    print("Qwen3.5-9B 다운로드 시작/재개 (최초 약 19.3GB). 진행 표시를 기다려주세요.")
    download_env = os.environ.copy()
    download_env["HF_HUB_OFFLINE"] = "0"
    download_env["TRANSFORMERS_OFFLINE"] = "0"
    download_env["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
    runner([sys.executable, "-u", "-c", DOWNLOAD_CODE, repo_id, str(Path(model_dir).resolve())],
           env=download_env, check=True)
    if not local_model_complete(model_dir):
        raise RuntimeError("모델 다운로드가 불완전합니다. 다운로드 오류를 확인하고 다시 Run All하세요.")
    return "downloaded"

try:
    download_status = ensure_model(MODEL_DIR, MODEL_REPO_ID)
except BaseException as exc:
    with open(os.path.join(OUTPUT_DIR, "setup_failure.json"), "w", encoding="utf-8") as f:
        json.dump({"stage": "model_download", "error": str(exc)}, f, ensure_ascii=False, indent=2)
    raise
revision_path = Path(MODEL_DIR) / "download_revision.json"
MODEL_REVISION = json.loads(revision_path.read_text(encoding="utf-8")).get("revision") if revision_path.is_file() else None
# 3) 다운로드 완료 후 학습·추론에서는 로컬 파일만 사용.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("준비 완료. 다음 셀부터 데이터 로드 → 학습 → 추론을 진행합니다.")

# 데이터 준비

데이터셋은 사전에 배포되어 `data` 폴더에 아래 구조로 준비되어 있어야 합니다. 다운로드·압축 해제 작업은 없습니다.

- data/train.csv, data/train 폴더
- data/test.csv, data/test 폴더
- data/sample_submission.csv


# 라이브러리, 데이터, 설정

In [ ]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    Qwen3_5ForConditionalGeneration,
    AutoProcessor,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

# 자동 다운로드/재사용 확인을 마친 로컬 Qwen3.5-9B 폴더
MODEL_ID = MODEL_DIR
IMAGE_SIZE = 512
MAX_NEW_TOKENS = 2  # 원본 generate 실제 길이 유지
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드 : 사전 배포된 data 폴더
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# 학습데이터 200개만 추출
train_df = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)

# 모델·Processor

다운로드된 원본 Qwen3.5-9B를 BF16(미지원 시 FP16)으로 로드합니다.
양자화 설정과 k-bit 학습 준비를 제거하고 LoRA 및 gradient checkpointing을 유지합니다.

In [ ]:
# 공식 모델 아키텍처와 로컬 파일 확인
assert torch.cuda.is_available(), "CUDA GPU가 필요합니다."
assert os.path.isfile(os.path.join(MODEL_ID, "config.json")), "앞의 자동 다운로드 셀을 먼저 실행하세요."
print("GPU VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GiB")

# 양자화 없이 원본 가중치를 직접 로드합니다.

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE*IMAGE_SIZE,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True,
    local_files_only=True,   # 로컬 파일만 사용
)

# 사전학습 모델
base_model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map={"": 0},
    dtype=COMPUTE_DTYPE,
    attn_implementation="sdpa",
    trust_remote_code=True,
    local_files_only=True,   # 로컬 파일만 사용
)

# 기본 가중치 동결은 get_peft_model에서 수행하며, LoRA만 학습합니다.
base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False
base_model.config.text_config.use_cache = False  # 학습 중 생성 캐시 비활성화

# LoRA 세팅
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
# 동결 임베딩 뒤의 checkpoint 경로에서도 LoRA까지 역전파되도록 활성값에 gradient를 연결.
# 임베딩 가중치 자체를 학습 대상으로 바꾸는 것은 아닙니다.
model.enable_input_require_grads()
if getattr(model, "is_loaded_in_4bit", False) or getattr(model, "is_loaded_in_8bit", False):
    raise RuntimeError("비양자화 실험에 양자화 모델이 로드되었습니다. 설정을 확인하세요.")
trainable_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
if not trainable_names or any("lora_" not in name for name in trainable_names):
    raise RuntimeError("학습 대상이 LoRA만으로 구성되어 있는지 확인하세요.")
model.print_trainable_parameters()

# 최소 재현 기록: 모델/패키지/데이터 설정과 실제 LoRA 파라미터 수
import json, hashlib, importlib.metadata as metadata
with open(os.path.join(DATA_DIR, "train.csv"), "rb") as f:
    train_csv_sha256 = hashlib.sha256(f.read()).hexdigest()
with open(os.path.join(MODEL_ID, "config.json"), "rb") as f:
    model_config_sha256 = hashlib.sha256(f.read()).hexdigest()
run_config = {
    "task_id": "TASK-003", "experiment_id": "EXP-010", "notebook_version": "v1.3", "model_id": "Qwen/Qwen3.5-9B",
    "model_path": os.path.abspath(MODEL_ID), "model_revision": MODEL_REVISION, "download_status": download_status, "seed": SEED, "sample_rows": 200,
    "train_rows": 180, "valid_rows": 20, "image_pixel_budget": IMAGE_SIZE**2,
    "max_new_tokens": MAX_NEW_TOKENS, "enable_thinking": False,
    "train_csv_sha256": train_csv_sha256, "model_config_sha256": model_config_sha256,
    "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "packages": {p: metadata.version(p) for p in ["torch", "transformers", "peft", "accelerate"]},
    "compute_dtype": str(COMPUTE_DTYPE),
    "quantization": None, "weight_precision_bits": 16,
    "base_weight_dtype": str(base_model.get_input_embeddings().weight.dtype),
    "parent_experiment_id": "EXP-004", "parent_reported_score": 0.83,
    "parent_notebook_sha256": "252345b05c30b911ec027f5ec3d78bdc65f29e8a9f4b142550e5efd726b4238c",
    "changes": ["image_pixel_budget_384_squared_to_512_squared", "nf4_to_unquantized_16bit"],
    "learning_rate": 1e-4, "epochs": 1, "batch_size": 1, "gradient_accumulation": 4,
    "lora_r": 8, "lora_alpha": 16, "lora_dropout": 0.05, "loss_scope": "full_sequence",
    "trainable_scope": "lora_only", "adapter_initialization": "new_from_original_base",
    "is_loaded_in_4bit": bool(getattr(model, "is_loaded_in_4bit", False)),
    "is_loaded_in_8bit": bool(getattr(model, "is_loaded_in_8bit", False)),
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_vram_bytes": torch.cuda.get_device_properties(0).total_memory,
    "status": "model_loaded_training_not_started"
}
with open(os.path.join(OUTPUT_DIR, "run_config.json"), "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [ ]:
# 모델 지시사항
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [ ]:
# 커스텀 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(os.path.join(DATA_DIR, row["path"])).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# 데이터 콜레이터
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
                enable_thinking=False
            )
            texts.append(text)
            images.append(img)

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if self.train:
            enc["labels"] = enc["input_ids"].clone()

        return enc

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [ ]:
# 검증용 데이터 분리
split = int(len(train_df)*0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# VQAMCDataset 형태로 변환
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# 데이터로더
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False, collate_fn=DataCollator(processor, True), num_workers=0)

# 현재 실행의 분할을 기록. 과거 3B 실행과 동일한지는 train.csv 해시/ID로 대조.
pd.concat([
    train_subset[["id", "path"]].assign(split="train"),
    valid_subset[["id", "path"]].assign(split="valid")
]).to_csv(os.path.join(OUTPUT_DIR, "split_manifest.csv"), index=False)

# Fine-tuning

기존과 동일한 180행·1 epoch, LoRA만 학습합니다.

In [ ]:
from tqdm.auto import tqdm

# 비양자화 모델은 로드 때 CUDA:0에 배치됨. LoRA만 학습.
GRAD_ACCUM = 4

# 옵티마이저, 학습 스케줄러
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
num_training_steps = 1 * math.ceil(len(train_loader)/GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)

# 스케일러
scaler = torch.amp.GradScaler("cuda", enabled=(COMPUTE_DTYPE == torch.float16))

# 학습 루프
import time
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
training_started = time.perf_counter()
global_step = 0
for epoch in range(1):
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        if epoch == 0 and step == 1:
            print("첫 배치 최대 할당 VRAM:", round(torch.cuda.max_memory_allocated() / 1024**3, 2), "GiB")
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0

    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad(), torch.autocast("cuda", dtype=COMPUTE_DTYPE):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"[Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")
    model.train()

torch.cuda.synchronize()
run_config["training_and_validation_seconds"] = time.perf_counter() - training_started
run_config["training_peak_allocated_bytes"] = torch.cuda.max_memory_allocated()
run_config["training_peak_reserved_bytes"] = torch.cuda.max_memory_reserved()

# 모델 저장
SAVE_DIR = os.path.join(OUTPUT_DIR, "qwen3_5_9b_lora")
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)

run_config["status"] = "training_completed"
run_config["valid_loss"] = val_loss / val_steps
run_config["adapter_path"] = os.path.abspath(SAVE_DIR)
with open(os.path.join(OUTPUT_DIR, "run_config.json"), "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

# Inference

512² 픽셀 예산과 비양자화 BF16/FP16 모델로 전체 test를 추론합니다.
기존 non-thinking + greedy 2토큰·생성 부분 디코딩·파싱 실패 시 a 정책을 유지합니다.
새 실행 폴더에 제출 CSV와 시간·메모리 기록을 저장합니다. 실제 점수·시간·VRAM은 실행 전입니다.

In [ ]:
# 데이터 파서 : 모델의 응답에서 선지를 추출
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

# 추론을 위해 모든 레이어 활성화
model.eval()
model.config.use_cache = True
model.config.text_config.use_cache = True
preds = []

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
inference_started = time.perf_counter()

# 추론 루프
for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(os.path.join(DATA_DIR, row["path"])).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad(), torch.autocast("cuda", dtype=COMPUTE_DTYPE):
        out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
    output_text = processor.batch_decode(out_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    # print("output_text:", output_text)
    # print("extract_choice:", extract_choice(output_text))
    preds.append(extract_choice(output_text))

torch.cuda.synchronize()
run_config["inference_seconds"] = time.perf_counter() - inference_started
run_config["inference_peak_allocated_bytes"] = torch.cuda.max_memory_allocated()

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(SUBMISSION_PATH, index=False)
print("Saved", SUBMISSION_PATH)
run_config["status"] = "test_inference_completed_submission_not_uploaded"
run_config["submission_path"] = os.path.abspath(SUBMISSION_PATH)
with open(os.path.join(OUTPUT_DIR, "run_config.json"), "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

In [ ]:
# 모델 응답 예시
print(output_text)